In [ ]:
from thesis_utils import APP_ROOT, DATA_ROOT, PREVIEW_ROWS, PROJECT_ROOT, paths
from pathlib import Path
import pandas as pd
import re

# ============================================================
# ANNOTATION EXCEL DOSYALARINDAKİ TÜM KOLON İSİMLERİNİ İNCELE
# ============================================================

ANNOTATION_DIR = paths.SP500_HUMAN_REVIEW_BATCHES_DIR

if not ANNOTATION_DIR.exists():
    raise FileNotFoundError(f"Klasör bulunamadı: {ANNOTATION_DIR}")

excel_files = [
    p for p in ANNOTATION_DIR.glob("*.xlsx")
    if not p.name.startswith("~$")
]

if not excel_files:
    raise FileNotFoundError(f"Excel dosyası bulunamadı: {ANNOTATION_DIR}")

def extract_batch_no(path):
    m = re.search(r"batch[_\s-]*(\d+)", path.stem.lower())
    if m:
        return int(m.group(1))
    return 999999

def find_header_row(path):
    raw = pd.read_excel(path, header=None, nrows=10)
    for idx in range(len(raw)):
        row_values = raw.iloc[idx].astype(str).str.strip().str.lower().tolist()
        if "annotation_id" in row_values:
            return idx
    return 0

excel_files = sorted(excel_files, key=lambda p: (extract_batch_no(p), p.name.lower()))

print("Bulunan Excel dosyası sayısı:", len(excel_files))
print("=" * 120)

all_columns_info = []

for file_path in excel_files:
    try:
        df_temp = pd.read_excel(file_path, nrows=5)

        print("\n" + "=" * 120)
        print("DOSYA:", file_path.name)
        print("Batch no:", extract_batch_no(file_path))
        print("Shape ilk okuma:", df_temp.shape)
        print("Kolonlar:")

        for i, col in enumerate(df_temp.columns):
            print(f"{i:02d} | {repr(col)}")

            all_columns_info.append({
                "file_name": file_path.name,
                "batch_no": extract_batch_no(file_path),
                "col_position": i,
                "column_name": str(col),
                "is_unnamed": str(col).lower().startswith("unnamed"),
            })

    except Exception as e:
        print("\nHATA:", file_path.name)
        print(e)

columns_df = pd.DataFrame(all_columns_info)

print("\n" + "=" * 120)
print("KOLON ANALİZİ ÖZETİ")
print("=" * 120)

print("\nUnique kolon isimleri:")
for col in sorted(columns_df["column_name"].unique()):
    print("-", repr(col))

print("\nUnnamed kolon sayısı:")
print(columns_df["is_unnamed"].value_counts(dropna=False))

print("\nDosya bazında Unnamed kolon sayısı:")
display(
    columns_df.groupby("file_name")["is_unnamed"]
    .sum()
    .reset_index()
    .rename(columns={"is_unnamed": "unnamed_column_count"})
    .sort_values("unnamed_column_count", ascending=False)
)

print("\nKolon pozisyonu bazında hangi isimler geliyor:")
display(
    columns_df.groupby("col_position")["column_name"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)

display(columns_df.head(PREVIEW_ROWS))

In [ ]:
from pathlib import Path
import pandas as pd
import re

# ============================================================
# ANNOTATION EXCEL DOSYALARINI TEKRAR OKU VE TEK DF YAP
# ============================================================

ANNOTATION_DIR = paths.SP500_HUMAN_REVIEW_BATCHES_DIR

if not ANNOTATION_DIR.exists():
    raise FileNotFoundError(f"Klasör bulunamadı: {ANNOTATION_DIR}")

# Sadece batch excel dosyalarını al
excel_files = [
    p for p in ANNOTATION_DIR.glob("SP500_annotation_batch_*.xlsx")
    if not p.name.startswith("~$")
]

if not excel_files:
    raise FileNotFoundError(f"Batch Excel dosyası bulunamadı: {ANNOTATION_DIR}")

def extract_batch_no(path):
    m = re.search(r"batch[_\s-]*(\d+)", path.stem.lower())
    if m:
        return int(m.group(1))
    return 999999

excel_files = sorted(excel_files, key=lambda p: extract_batch_no(p))

print("Okunacak dosya sayısı:", len(excel_files))
print("İlk 3 dosya:")
for p in excel_files[:PREVIEW_ROWS]:
    print("-", p.name)

print("\nSon 3 dosya:")
for p in excel_files[-PREVIEW_ROWS:]:
    print("-", p.name)


# ------------------------------------------------------------
# 1) Dosyaları oku
# ------------------------------------------------------------
dfs = []

for file_path in excel_files:
    temp = pd.read_excel(file_path, header=find_header_row(file_path))

    # Tamamen boş kolonları sil
    temp = temp.dropna(axis=1, how="all")

    # Tamamen boş satırları sil
    temp = temp.dropna(axis=0, how="all")

    # Kolon isimlerini temizle
    temp.columns = [str(c).strip() for c in temp.columns]

    # Sağ taraftaki özet/boş kolonları at
    drop_cols = [
        c for c in temp.columns
        if c.startswith("Unnamed")
        or c in ["Özet", "Değer"]
    ]

    if drop_cols:
        temp = temp.drop(columns=drop_cols)

    temp["source_excel_file"] = file_path.name
    temp["batch_no"] = extract_batch_no(file_path)

    dfs.append(temp)

annotation_all_df = pd.concat(dfs, ignore_index=True)


# ------------------------------------------------------------
# 2) Temel temizlik
# ------------------------------------------------------------
# Tamamen boş satırları tekrar at
annotation_all_df = annotation_all_df.dropna(axis=0, how="all").copy()

# annotation_id boş olan satırları at
if "annotation_id" in annotation_all_df.columns:
    annotation_all_df = annotation_all_df[
        annotation_all_df["annotation_id"].notna()
    ].copy()

# String kolonları temizle
string_cols = [
    "annotation_id",
    "sample_id",
    "text_en",
    "text_tr",
    "finbert_label",
    "chatgpt_label",
    "chatgpt_confidence",
    "chatgpt_reason_tr",
    "final_label",
    "finbert_correctness",
    "finbert_correctness_note",
]

for col in string_cols:
    if col in annotation_all_df.columns:
        annotation_all_df[col] = annotation_all_df[col].astype("string").str.strip()

# Tarih ve numeric düzeltme
if "date" in annotation_all_df.columns:
    annotation_all_df["date"] = pd.to_datetime(annotation_all_df["date"], errors="coerce")

if "finbert_confidence" in annotation_all_df.columns:
    annotation_all_df["finbert_confidence"] = pd.to_numeric(
        annotation_all_df["finbert_confidence"],
        errors="coerce"
    )


# ------------------------------------------------------------
# 3) annotation_id sırasına göre sırala
# ------------------------------------------------------------
def extract_ann_no(x):
    m = re.search(r"(\d+)", str(x))
    if m:
        return int(m.group(1))
    return 999999999

annotation_all_df["_ann_no"] = annotation_all_df["annotation_id"].apply(extract_ann_no)

annotation_all_df = annotation_all_df.sort_values(
    ["_ann_no", "batch_no"]
).drop(columns=["_ann_no"]).reset_index(drop=True)


# ------------------------------------------------------------
# 4) Özet kontrol
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("MERGED ANNOTATION DF READY")
print("=" * 100)

print("Okunan dosya sayısı:", len(excel_files))
print("Toplam satır:", len(annotation_all_df))
print("Shape:", annotation_all_df.shape)

print("\nKolonlar:")
print(annotation_all_df.columns.tolist())

print("\nİlk annotation_id:", annotation_all_df["annotation_id"].head(PREVIEW_ROWS).tolist())
print("Son annotation_id:", annotation_all_df["annotation_id"].tail(PREVIEW_ROWS).tolist())

print("\nBoşluk kontrolü:")
for col in ["annotation_id", "sample_id", "text_en", "text_tr", "chatgpt_label", "final_label"]:
    if col in annotation_all_df.columns:
        print(col, "boş:", annotation_all_df[col].isna().sum())

print("\nfinal_label dağılımı:")
if "final_label" in annotation_all_df.columns:
    print(annotation_all_df["final_label"].value_counts(dropna=False))

print("\nchatgpt_label dağılımı:")
if "chatgpt_label" in annotation_all_df.columns:
    print(annotation_all_df["chatgpt_label"].value_counts(dropna=False))

print("\nfinbert_label dağılımı:")
if "finbert_label" in annotation_all_df.columns:
    print(annotation_all_df["finbert_label"].value_counts(dropna=False))

print("\nfinbert_correctness dağılımı:")
if "finbert_correctness" in annotation_all_df.columns:
    print(annotation_all_df["finbert_correctness"].value_counts(dropna=False))

#display(annotation_all_df.head(PREVIEW_ROWS))
#display(annotation_all_df.tail(PREVIEW_ROWS))

In [ ]:
annotation_all_df.tail(2)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
import pandas as pd
import numpy as np

# ============================================================
# FINBERT PERFORMANCE ON annotation_all_df
# ============================================================

df_eval = annotation_all_df.copy()

# ------------------------------------------------------------
# 1) Hangi label kolonunu kullanacağız?
# ------------------------------------------------------------
# Öncelik final_label. Eğer final_label boşsa chatgpt_label kullan.
if "final_label" in df_eval.columns:
    df_eval["gold_label"] = df_eval["final_label"]
else:
    df_eval["gold_label"] = pd.NA

if "chatgpt_label" in df_eval.columns:
    df_eval["gold_label"] = df_eval["gold_label"].fillna(df_eval["chatgpt_label"])

# String normalize
df_eval["gold_label"] = (
    df_eval["gold_label"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df_eval["finbert_label"] = (
    df_eval["finbert_label"]
    .astype(str)
    .str.lower()
    .str.strip()
)

valid_labels = ["negative", "neutral", "positive"]

# Geçerli satırları al
df_eval = df_eval[
    df_eval["gold_label"].isin(valid_labels) &
    df_eval["finbert_label"].isin(valid_labels)
].copy()

print("Eval shape:", df_eval.shape)

print("\nGold label dağılımı:")
print(df_eval["gold_label"].value_counts())

print("\nFinBERT prediction dağılımı:")
print(df_eval["finbert_label"].value_counts())

# ------------------------------------------------------------
# 2) Label id mapping
# ------------------------------------------------------------
LABEL_ORDER = ["negative", "neutral", "positive"]

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

y_true = df_eval["gold_label"].map(label2id).astype(int)
y_pred = df_eval["finbert_label"].map(label2id).astype(int)

# ------------------------------------------------------------
# 3) Genel metrikler
# ------------------------------------------------------------
acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average="macro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")

precision_macro, recall_macro, _, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

precision_weighted, recall_weighted, _, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

metrics_df = pd.DataFrame([{
    "model": "Original FinBERT",
    "test_set": "SP500 annotation_all_df",
    "n_eval": len(df_eval),
    "accuracy": acc,
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
    "f1_macro": f1_macro,
    "precision_weighted": precision_weighted,
    "recall_weighted": recall_weighted,
    "f1_weighted": f1_weighted,
}])

print("\n" + "=" * 100)
print("FINBERT METRICS ON annotation_all_df")
print("=" * 100)
display(metrics_df.round(4))

# ------------------------------------------------------------
# 4) Classification report
# ------------------------------------------------------------
print("\nClassification Report:")
report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=LABEL_ORDER,
    digits=4,
    zero_division=0
)
print(report)

# ------------------------------------------------------------
# 5) Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2]
)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{x}" for x in LABEL_ORDER],
    columns=[f"pred_{x}" for x in LABEL_ORDER]
)

print("\nConfusion Matrix:")
display(cm_df)

# Normalize edilmiş confusion matrix
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

cm_norm_df = pd.DataFrame(
    cm_norm,
    index=[f"true_{x}" for x in LABEL_ORDER],
    columns=[f"pred_{x}" for x in LABEL_ORDER]
)

print("\nConfusion Matrix - Row Normalized:")
display(cm_norm_df.round(3))

# ------------------------------------------------------------
# 6) Doğru / yanlış analizi
# ------------------------------------------------------------
df_eval["finbert_is_correct"] = df_eval["gold_label"] == df_eval["finbert_label"]

print("\nFinBERT correct / wrong:")
print(df_eval["finbert_is_correct"].value_counts())

print("\nFinBERT correct ratio:")
print(df_eval["finbert_is_correct"].value_counts(normalize=True).mul(100).round(2))

print("\nDoğru/yanlış label bazında:")
display(
    df_eval.groupby("gold_label")["finbert_is_correct"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "accuracy_by_true_label"})
    .sort_index()
)

# ------------------------------------------------------------
# 7) Confidence analizi
# ------------------------------------------------------------
if "finbert_confidence" in df_eval.columns:
    df_eval["finbert_confidence"] = pd.to_numeric(
        df_eval["finbert_confidence"],
        errors="coerce"
    )

    print("\nFinBERT confidence summary:")
    display(df_eval["finbert_confidence"].describe())

    print("\nConfidence by correct/wrong:")
    display(
        df_eval.groupby("finbert_is_correct")["finbert_confidence"]
        .describe()
    )

    print("\nConfidence by true label:")
    display(
        df_eval.groupby("gold_label")["finbert_confidence"]
        .describe()
    )

# ------------------------------------------------------------
# 8) En yüksek confidence ile yanlış yaptığı örnekler
# ------------------------------------------------------------
wrong_df = df_eval[df_eval["finbert_is_correct"] == False].copy()

if "finbert_confidence" in wrong_df.columns:
    wrong_df = wrong_df.sort_values("finbert_confidence", ascending=False)

print("\nWrong predictions:", wrong_df.shape)

show_cols = [
    "annotation_id",
    "sample_id",
    "date",
    "text_en",
    "text_tr",
    "gold_label",
    "finbert_label",
    "finbert_confidence",
    "chatgpt_reason_tr",
]

show_cols = [c for c in show_cols if c in wrong_df.columns]

print("\nHigh-confidence wrong examples:")
display(wrong_df[show_cols].head(PREVIEW_ROWS))



In [ ]:
# ============================================================
# TRAIN EDİLMİŞ MODELLERİ annotation_all_df ÜZERİNDE TEST ET
# KAYIT YOK - SADECE EKRANA BASAR
#
# Modeller:
# - original_finbert: annotation_all_df içindeki finbert_label
# - roberta_base
# - bert_base_uncased
# - distilbert_base_uncased
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 0) Ayarlar
# ------------------------------------------------------------
PROJECT_DIR = PROJECT_ROOT
CHECKPOINT_ROOT = PROJECT_DIR / "checkpoints" / "financial_sentiment_multi_model"

VALID_LABELS = ["negative", "neutral", "positive"]

LABEL2ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

ID2LABEL = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

MODEL_RUNS = [
    {
        "run_name": "roberta_base",
        "model_dir": CHECKPOINT_ROOT / "roberta_base" / "final_model",
    },
    {
        "run_name": "bert_base_uncased",
        "model_dir": CHECKPOINT_ROOT / "bert_base_uncased" / "final_model",
    },
    {
        "run_name": "distilbert_base_uncased",
        "model_dir": CHECKPOINT_ROOT / "distilbert_base_uncased" / "final_model",
    },
]

MAX_LENGTH = 128
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# ------------------------------------------------------------
# 1) Eval dataframe hazırla
# ------------------------------------------------------------
if "annotation_all_df" not in globals():
    raise ValueError("annotation_all_df bulunamadı. Önce Excel dosyalarını birleştiren hücreyi çalıştır.")

df_eval = annotation_all_df.copy()

# Text kolonu
if "text_en" in df_eval.columns:
    df_eval["eval_text"] = df_eval["text_en"]
elif "text" in df_eval.columns:
    df_eval["eval_text"] = df_eval["text"]
else:
    raise ValueError("annotation_all_df içinde text_en veya text kolonu bulunamadı.")

df_eval["eval_text"] = (
    df_eval["eval_text"]
    .astype("string")
    .str.strip()
)

# Gold label: final_label varsa onu kullan, boşsa chatgpt_label
if "final_label" in df_eval.columns:
    df_eval["gold_label"] = df_eval["final_label"]
else:
    df_eval["gold_label"] = pd.NA

df_eval["gold_label"] = (
    df_eval["gold_label"]
    .astype("string")
    .str.lower()
    .str.strip()
    .replace(["", "nan", "none", "<na>"], pd.NA)
)

if "chatgpt_label" in df_eval.columns:
    chatgpt_label = (
        df_eval["chatgpt_label"]
        .astype("string")
        .str.lower()
        .str.strip()
        .replace(["", "nan", "none", "<na>"], pd.NA)
    )
    df_eval["gold_label"] = df_eval["gold_label"].fillna(chatgpt_label)

df_eval = df_eval[
    df_eval["eval_text"].notna() &
    (df_eval["eval_text"] != "") &
    df_eval["gold_label"].isin(VALID_LABELS)
].copy()

df_eval = df_eval.reset_index(drop=True)
df_eval["gold_id"] = df_eval["gold_label"].map(LABEL2ID).astype(int)

print("\nEval shape:", df_eval.shape)

print("\nGold label distribution:")
print(df_eval["gold_label"].value_counts())

print("\nGold label ratio:")
print(df_eval["gold_label"].value_counts(normalize=True).mul(100).round(2))

display(df_eval[["annotation_id", "sample_id", "eval_text", "gold_label"]].head(PREVIEW_ROWS))

# ------------------------------------------------------------
# 2) Metrik fonksiyonları
# ------------------------------------------------------------
def compute_full_metrics(y_true, y_pred, model_name, test_set_name):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    precision_macro, recall_macro, _, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, _, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    return {
        "model": model_name,
        "test_set": test_set_name,
        "n_eval": int(len(y_true)),
        "accuracy": float(acc),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
    }


def make_report_and_cm(y_true, y_pred):
    report_text = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=VALID_LABELS,
        digits=4,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2]
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{x}" for x in VALID_LABELS],
        columns=[f"pred_{x}" for x in VALID_LABELS],
    )

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    cm_norm_df = pd.DataFrame(
        cm_norm,
        index=[f"true_{x}" for x in VALID_LABELS],
        columns=[f"pred_{x}" for x in VALID_LABELS],
    )

    return report_text, cm_df, cm_norm_df


def print_model_result(model_name, y_true, y_pred):
    metrics = compute_full_metrics(
        y_true=y_true,
        y_pred=y_pred,
        model_name=model_name,
        test_set_name="SP500_annotation_all"
    )

    report_text, cm_df, cm_norm_df = make_report_and_cm(y_true, y_pred)

    print("\n" + "=" * 100)
    print(f"{model_name} RESULT")
    print("=" * 100)

    display(pd.DataFrame([metrics]).round(4))

    print("\nClassification Report:")
    print(report_text)

    print("\nConfusion Matrix:")
    display(cm_df)

    print("\nConfusion Matrix Normalized:")
    display(cm_norm_df.round(3))

    return metrics

# ------------------------------------------------------------
# 3) Original FinBERT baseline
# ------------------------------------------------------------
all_metrics = []

if "finbert_label" in df_eval.columns:
    finbert_eval = df_eval.copy()

    finbert_eval["finbert_label_norm"] = (
        finbert_eval["finbert_label"]
        .astype("string")
        .str.lower()
        .str.strip()
    )

    finbert_eval = finbert_eval[
        finbert_eval["finbert_label_norm"].isin(VALID_LABELS)
    ].copy()

    y_true_finbert = finbert_eval["gold_label"].map(LABEL2ID).astype(int).values
    y_pred_finbert = finbert_eval["finbert_label_norm"].map(LABEL2ID).astype(int).values

    metrics = print_model_result(
        model_name="original_finbert",
        y_true=y_true_finbert,
        y_pred=y_pred_finbert
    )

    all_metrics.append(metrics)

# ------------------------------------------------------------
# 4) Train edilmiş model prediction fonksiyonu
# ------------------------------------------------------------
@torch.no_grad()
def predict_with_model(model_dir, texts, batch_size=32, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)

    model.to(device)
    model.eval()

    pred_ids_all = []
    pred_labels = []
    confidences = []

    texts = [str(x) for x in texts]

    for start in tqdm(range(0, len(texts), batch_size), desc=f"Predicting {model_dir.parent.name}"):
        batch_texts = texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()
        pred_ids = probs.argmax(axis=1)

        for pred_id, prob_vec in zip(pred_ids, probs):
            pred_id = int(pred_id)

            pred_ids_all.append(pred_id)
            pred_labels.append(ID2LABEL[pred_id])
            confidences.append(float(prob_vec[pred_id]))

    del model
    del tokenizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame({
        "pred_id": pred_ids_all,
        "pred_label": pred_labels,
        "pred_confidence": confidences,
    })

# ------------------------------------------------------------
# 5) Train edilmiş modelleri sırayla test et
# ------------------------------------------------------------
all_prediction_dfs = {}

for run in MODEL_RUNS:
    run_name = run["run_name"]
    model_dir = Path(run["model_dir"])

    if not model_dir.exists():
        print("\nUYARI: Model klasörü bulunamadı:", model_dir)
        continue

    print("\n" + "#" * 120)
    print(f"MODEL TEST EDİLİYOR: {run_name}")
    print("Model dir:", model_dir)
    print("#" * 120)

    pred_df = predict_with_model(
        model_dir=model_dir,
        texts=df_eval["eval_text"].tolist(),
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH
    )

    y_true = df_eval["gold_id"].astype(int).values
    y_pred = pred_df["pred_id"].astype(int).values

    metrics = print_model_result(
        model_name=run_name,
        y_true=y_true,
        y_pred=y_pred
    )

    all_metrics.append(metrics)

    model_eval_df = pd.concat(
        [
            df_eval[["annotation_id", "sample_id", "eval_text", "gold_label", "gold_id"]].reset_index(drop=True),
            pred_df.reset_index(drop=True)
        ],
        axis=1
    )

    model_eval_df["is_correct"] = model_eval_df["gold_label"] == model_eval_df["pred_label"]

    print("\nCorrect / Wrong:")
    print(model_eval_df["is_correct"].value_counts())
    print(model_eval_df["is_correct"].value_counts(normalize=True).mul(100).round(2))

    print("\nAccuracy by true label:")
    display(
        model_eval_df.groupby("gold_label")["is_correct"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "accuracy_by_label"})
        .sort_index()
    )

    all_prediction_dfs[run_name] = model_eval_df

# ------------------------------------------------------------
# 6) Summary tablo
# ------------------------------------------------------------
summary_df = pd.DataFrame(all_metrics)

summary_df = summary_df.sort_values(
    "f1_macro",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 120)
print("FINAL SUMMARY ON SP500 ANNOTATION SET")
print("=" * 120)

display(summary_df.round(4))

# 05 - Fine-Tuned Modellerin S&P 500 Dış Testi

## 1. Dış Test Setinin Amacı

Bu aşamada modeller, eğitim ve validation sürecinde kullanılmayan bağımsız bir test seti üzerinde değerlendirilmiştir. Bu test seti, S&P 500 finansal haber başlıklarından oluşturulan ve manuel/ChatGPT destekli olarak etiketlenen annotation setidir.

Bu dış testin amacı, modellerin yalnızca eğitim veri setinin kendi test bölümünde değil, farklı bir finansal haber başlığı veri setinde de ne kadar genelleme yapabildiğini ölçmektir.

Değerlendirilen modeller:

- Original FinBERT
- RoBERTa-base fine-tuned
- BERT-base-uncased fine-tuned
- DistilBERT-base-uncased fine-tuned

Toplam dış test örneği:

| Test Seti | Örnek Sayısı |
|---|---:|
| S&P 500 Annotation Set | 960 |

---

## 2. Dış Test Setindeki Sınıf Dağılımı

| Sınıf | Örnek Sayısı |
|---|---:|
| Negative | 293 |
| Neutral | 310 |
| Positive | 357 |
| **Toplam** | **960** |

Bu dağılım, önceki eğitim/test veri setine göre daha dengeli bir yapı göstermektedir. Bu nedenle accuracy metriği daha anlamlı hale gelse de, model karşılaştırmasında yine ana metrik olarak **macro-F1** kullanılmıştır.

---

## 3. Genel Model Karşılaştırması

| Sıra | Model | Accuracy | Precision Macro | Recall Macro | Macro-F1 | Weighted-F1 |
|---:|---|---:|---:|---:|---:|---:|
| 1 | **RoBERTa-base** | **0.8125** | **0.8240** | **0.8126** | **0.8138** | **0.8146** |
| 2 | Original FinBERT | 0.7583 | 0.7596 | 0.7626 | 0.7585 | 0.7579 |
| 3 | DistilBERT-base-uncased | 0.7531 | 0.7686 | 0.7519 | 0.7523 | 0.7536 |
| 4 | BERT-base-uncased | 0.7521 | 0.7677 | 0.7514 | 0.7511 | 0.7524 |

Dış test sonuçlarına göre en iyi model **RoBERTa-base** olmuştur. RoBERTa-base modeli, hazır FinBERT baseline’ını bağımsız S&P 500 annotation setinde de geçmiştir.

---

## 4. RoBERTa-base ve FinBERT Karşılaştırması

| Model | Accuracy | Macro-F1 | Weighted-F1 |
|---|---:|---:|---:|
| Original FinBERT | 0.7583 | 0.7585 | 0.7579 |
| **RoBERTa-base** | **0.8125** | **0.8138** | **0.8146** |

RoBERTa-base modelinin FinBERT’e göre performans artışı:

| Metrik | Artış |
|---|---:|
| Accuracy | +0.0542 |
| Macro-F1 | +0.0553 |
| Weighted-F1 | +0.0567 |

Bu sonuç, RoBERTa-base modelinin yalnızca iç test setinde değil, bağımsız S&P 500 haber başlıkları üzerinde de FinBERT baseline’ından daha yüksek performans gösterdiğini ortaya koymaktadır.

---

## 5. Original FinBERT Dış Test Sonucu

### Genel Metrikler

| Metrik | Değer |
|---|---:|
| Accuracy | 0.7583 |
| Precision Macro | 0.7596 |
| Recall Macro | 0.7626 |
| Macro-F1 | 0.7585 |
| Weighted-F1 | 0.7579 |

### Sınıf Bazlı Sonuçlar

| Sınıf | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| Negative | 0.7259 | 0.8225 | 0.7712 | 293 |
| Neutral | 0.7391 | 0.7677 | 0.7532 | 310 |
| Positive | 0.8137 | 0.6975 | 0.7511 | 357 |

### Confusion Matrix

| Gerçek / Tahmin | Pred Negative | Pred Neutral | Pred Positive |
|---|---:|---:|---:|
| True Negative | 241 | 34 | 18 |
| True Neutral | 33 | 238 | 39 |
| True Positive | 58 | 50 | 249 |

### Yorum

FinBERT, negative sınıfında yüksek recall değeri elde etmiştir. Gerçek negative olan 293 örneğin 241 tanesini doğru sınıflandırmıştır. Ancak positive sınıfında recall değeri 0.6975 ile daha düşük kalmıştır. Bu durum, FinBERT’in bazı olumlu piyasa yönelimli haber başlıklarını negative veya neutral sınıflara karıştırdığını göstermektedir.

Özellikle:

- True positive → pred negative: 58
- True positive → pred neutral: 50

Bu hata tipi, FinBERT’in bazı olumlu piyasa sinyallerini yeterince iyi yakalayamadığını göstermektedir.

---

## 6. RoBERTa-base Dış Test Sonucu

### Genel Metrikler

| Metrik | Değer |
|---|---:|
| Accuracy | 0.8125 |
| Precision Macro | 0.8240 |
| Recall Macro | 0.8126 |
| Macro-F1 | 0.8138 |
| Weighted-F1 | 0.8146 |

### Sınıf Bazlı Sonuçlar

| Sınıf | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| Negative | 0.8906 | 0.7782 | 0.8306 | 293 |
| Neutral | 0.7045 | 0.8613 | 0.7750 | 310 |
| Positive | 0.8769 | 0.7983 | 0.8358 | 357 |

### Confusion Matrix

| Gerçek / Tahmin | Pred Negative | Pred Neutral | Pred Positive |
|---|---:|---:|---:|
| True Negative | 228 | 50 | 15 |
| True Neutral | 18 | 267 | 25 |
| True Positive | 10 | 62 | 285 |

### Yorum

RoBERTa-base modeli dış test setinde en yüksek başarıyı elde etmiştir. Özellikle negative ve positive sınıflarında yüksek precision değerleri dikkat çekmektedir.

- Negative precision: 0.8906
- Positive precision: 0.8769

Bu durum, RoBERTa’nın negative veya positive tahmini yaptığında genellikle doğru tahmin yaptığını göstermektedir.

RoBERTa’nın en zayıf tarafı neutral precision değeridir. Model bazı gerçek negative ve positive örnekleri neutral sınıfına kaydırmıştır.

Başlıca hata tipleri:

- True negative → pred neutral: 50
- True positive → pred neutral: 62

Bu durum, modelin bazı yönlü haber başlıklarında yeterince güçlü sinyal görmediğinde neutral sınıfına yöneldiğini göstermektedir.

---

## 7. BERT-base-uncased Dış Test Sonucu

### Genel Metrikler

| Metrik | Değer |
|---|---:|
| Accuracy | 0.7521 |
| Precision Macro | 0.7677 |
| Recall Macro | 0.7514 |
| Macro-F1 | 0.7511 |
| Weighted-F1 | 0.7524 |

### Sınıf Bazlı Sonuçlar

| Sınıf | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| Negative | 0.8326 | 0.6621 | 0.7376 | 293 |
| Neutral | 0.6568 | 0.8581 | 0.7441 | 310 |
| Positive | 0.8137 | 0.7339 | 0.7717 | 357 |

### Confusion Matrix

| Gerçek / Tahmin | Pred Negative | Pred Neutral | Pred Positive |
|---|---:|---:|---:|
| True Negative | 194 | 69 | 30 |
| True Neutral | 14 | 266 | 30 |
| True Positive | 25 | 70 | 262 |

### Yorum

BERT-base-uncased modeli iç test setinde güçlü performans göstermesine rağmen, bağımsız S&P 500 annotation setinde FinBERT baseline’ın biraz gerisinde kalmıştır.

Model özellikle neutral sınıfında yüksek recall elde etmiştir:

- Neutral recall: 0.8581

Ancak negative sınıfında recall değeri düşüktür:

- Negative recall: 0.6621

Bu durum, BERT-base modelinin bağımsız dış test setinde bazı negative örnekleri neutral sınıfına kaydırdığını göstermektedir.

---

## 8. DistilBERT-base-uncased Dış Test Sonucu

### Genel Metrikler

| Metrik | Değer |
|---|---:|
| Accuracy | 0.7531 |
| Precision Macro | 0.7686 |
| Recall Macro | 0.7519 |
| Macro-F1 | 0.7523 |
| Weighted-F1 | 0.7536 |

### Sınıf Bazlı Sonuçlar

| Sınıf | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| Negative | 0.8398 | 0.6621 | 0.7405 | 293 |
| Neutral | 0.6575 | 0.8484 | 0.7408 | 310 |
| Positive | 0.8085 | 0.7451 | 0.7755 | 357 |

### Confusion Matrix

| Gerçek / Tahmin | Pred Negative | Pred Neutral | Pred Positive |
|---|---:|---:|---:|
| True Negative | 194 | 69 | 30 |
| True Neutral | 14 | 263 | 33 |
| True Positive | 23 | 68 | 266 |

### Yorum

DistilBERT-base-uncased modeli BERT-base’e çok yakın bir dış test performansı göstermiştir. Hatta macro-F1 açısından BERT-base’in az da olsa üzerinde yer almıştır.

Ancak DistilBERT de FinBERT ve RoBERTa’nın gerisinde kalmıştır. Buna rağmen DistilBERT daha hafif ve hızlı bir model olduğu için pratik kullanım açısından alternatif olarak değerlendirilebilir.

---

## 9. İç Test ve Dış Test Karşılaştırması

İç test setinde elde edilen macro-F1 sonuçları:

| Model | İç Test Macro-F1 |
|---|---:|
| RoBERTa-base | 0.8651 |
| BERT-base-uncased | 0.8378 |
| DistilBERT-base-uncased | 0.8218 |
| Original FinBERT | 0.6794 |

Bağımsız S&P 500 annotation setinde elde edilen macro-F1 sonuçları:

| Model | Dış Test Macro-F1 |
|---|---:|
| RoBERTa-base | 0.8138 |
| Original FinBERT | 0.7585 |
| DistilBERT-base-uncased | 0.7523 |
| BERT-base-uncased | 0.7511 |

İç test setinde fine-tune edilen tüm Transformer modelleri FinBERT’i geçmiştir. Ancak bağımsız S&P 500 annotation setinde yalnızca RoBERTa-base modeli FinBERT baseline’ından daha yüksek performans göstermiştir.

Bu durum, modellerin genelleme başarısı açısından RoBERTa-base modelinin daha güçlü olduğunu göstermektedir.

---

## 10. Dış Test Sonuçlarının Yorumu

Dış test sonuçları şu açıdan önemlidir:

1. S&P 500 annotation seti eğitim verisinden bağımsızdır.
2. Bu set farklı bir kaynak ve haber başlığı formatı içermektedir.
3. Dolayısıyla bu test, modellerin genelleme performansını daha gerçekçi biçimde ölçmektedir.
4. RoBERTa-base modeli bu dış testte de FinBERT’i geçmiştir.
5. BERT-base ve DistilBERT modelleri iç testte başarılı olsa da dış testte FinBERT’in biraz gerisinde kalmıştır.

Bu sonuç, yalnızca hedef veri setine uyum sağlamanın değil, farklı finansal haber başlıklarına genelleme yapabilmenin de önemli olduğunu göstermektedir.

---

## 11. Nihai Model Seçimi

Dış test sonuçlarına göre nihai model olarak **RoBERTa-base** öne çıkmaktadır.

Bunun nedenleri:

1. İç test setinde en yüksek macro-F1 değerini elde etmiştir.
2. Bağımsız S&P 500 annotation setinde de en yüksek macro-F1 değerine ulaşmıştır.
3. Hazır FinBERT baseline’ını dış testte yaklaşık 5.5 puan macro-F1 farkıyla geçmiştir.
4. Negative ve positive sınıflarında yüksek precision değerleri elde etmiştir.
5. Genel genelleme performansı diğer fine-tune modellerden daha güçlüdür.

---

## 12. Ana Bulgular

Bu dış test deneyinden elde edilen temel bulgular şunlardır:

1. Bağımsız S&P 500 annotation setinde en iyi sonuç RoBERTa-base modeliyle elde edilmiştir.
2. RoBERTa-base, 0.8125 accuracy ve 0.8138 macro-F1 değerlerine ulaşmıştır.
3. Hazır FinBERT aynı sette 0.7583 accuracy ve 0.7585 macro-F1 elde etmiştir.
4. RoBERTa-base modeli, FinBERT baseline’ını dış testte de geçmiştir.
5. BERT-base ve DistilBERT modelleri iç test setinde güçlü sonuçlar üretmiş olsa da dış testte FinBERT’in biraz altında kalmıştır.
6. Bu sonuçlar, RoBERTa-base modelinin daha iyi genelleme yaptığını göstermektedir.
7. Nihai model olarak RoBERTa-base seçilmesi daha uygundur.

---

## 13. Tez İçin Kullanılabilecek Sonuç Paragrafı

Bağımsız S&P 500 haber başlıklarından oluşturulan 960 örneklik annotation seti üzerinde yapılan değerlendirmede, RoBERTa-base modeli en yüksek performansı göstermiştir. RoBERTa-base modeli bu sette 0.8125 accuracy, 0.8138 macro-F1 ve 0.8146 weighted-F1 değerlerine ulaşmıştır. Hazır FinBERT modeli ise aynı sette 0.7583 accuracy, 0.7585 macro-F1 ve 0.7579 weighted-F1 elde etmiştir. BERT-base-uncased ve DistilBERT-base-uncased modelleri sırasıyla 0.7511 ve 0.7523 macro-F1 değerlerinde kalmıştır. Bu sonuçlar, RoBERTa-base modelinin yalnızca hedef veri setinin iç test bölümünde değil, bağımsız finansal haber başlıkları üzerinde de FinBERT baseline’ından daha yüksek genelleme performansı sağladığını göstermektedir.

---



## 14. Genel Sonuç

Bu dış test sonuçları, çalışmanın temel bulgusunu güçlendirmektedir. Hedef veri seti üzerinde fine-tune edilen RoBERTa-base modeli, hazır FinBERT baseline’ını hem iç test setinde hem de bağımsız S&P 500 annotation setinde geçmiştir.

Bu nedenle çalışmanın nihai model seçimi:

**RoBERTa-base**

Ana sonuç:

**Hedef veri setine özel fine-tuning, doğru model mimarisiyle birleştiğinde hazır FinBERT kullanımına göre daha yüksek performans ve daha iyi genelleme sağlayabilmektedir.**


##  Kısa Özet

Modelleri bağımsız olarak oluşturduğumuz 960 örneklik S&P 500 annotation setinde de test ettim. Bu dış testte en iyi sonucu RoBERTa-base verdi. RoBERTa 0.8125 accuracy ve 0.8138 macro-F1 elde etti. Hazır FinBERT ise aynı sette 0.7583 accuracy ve 0.7585 macro-F1 aldı. Yani RoBERTa dış testte de FinBERT’i yaklaşık 5.5 puan macro-F1 farkıyla geçti.

BERT-base ve DistilBERT modelleri iç test setinde iyi sonuç vermelerine rağmen dış testte FinBERT’in biraz altında kaldı. Bu yüzden nihai model olarak RoBERTa-base daha güçlü görünüyor. Sonuçlar, RoBERTa’nın yalnızca eğitim veri dağılımına değil, bağımsız S&P 500 haber başlıklarına da daha iyi genelleme yaptığını gösteriyor.

---